[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟡 Medium: Top-k / Top-p Sampling

*Inference & Decoding*
Implement the standard **LLM sampling head**: temperature scaling, top-k
truncation, top-p (nucleus) truncation, and a single categorical draw.

```python
def sample_top_k_top_p(key, logits, *, temperature=1.0, top_k=None, top_p=1.0):
    ...  # -> scalar int32 token id
```

### The three knobs

| knob | what it does | effect |
|---|---|---|
| `temperature` | $z \leftarrow z / T$ | $T<1$ sharpens, $T>1$ flattens, $T\to0$ is argmax |
| `top_k` | keep the $k$ largest logits | fixed-size support |
| `top_p` | keep the smallest prefix of the sorted distribution with mass $\ge p$ | adaptive support |

Top-p keeps the **smallest** set $S$ of highest-probability tokens with
$\sum_{x \in S} p(x) \ge p$. Concretely, sort descending and keep rank $i$ iff

$$\sum_{j<i} p_{(j)} \;<\; p$$

The *exclusive* cumulative sum is what makes the token that crosses the
threshold get kept, and it keeps rank 0 alive for any `top_p` $> 0$.
`top_p = 0` is the one case it does not cover — force rank 0 in explicitly so
the filter can never mask the whole vocabulary.

### Rules
- Everything not kept is set to `-jnp.inf` **before** the draw — do not
  renormalise by hand, `jax.random.categorical` takes logits
- Apply in the order **temperature → top-k → top-p → sample**
- top-p sees the already-top-k-masked logits (the two filters compose)
- At least one token must always survive
- No Python loop over the vocabulary; mask with `jnp.where`
- Must work under `jax.jit` (with `top_k`/`top_p` as static Python values) and
  under `jax.vmap` over a batch of keys
- `logits` is 1-D of shape `(V,)`; return a **scalar** integer array

### Why the order matters (the interview question)
Temperature commutes with top-k — dividing by $T$ is monotone, so the identity
of the $k$ largest logits never changes. It does **not** commute with top-p:
the nucleus is defined on probabilities, and $T$ changes them. With
$z = [3,2,1,0]$ and $p = 0.9$, $T=1$ admits three tokens but $T=0.5$ admits
only two. Filtering before scaling gives you the nucleus of the *unscaled*
distribution — wider than you asked for whenever $T<1$, narrower whenever
$T>1$. Ship that bug and your "deterministic, low-temperature" endpoint keeps
emitting tokens the user thought they had truncated away.

The second trap is renormalising twice. Masking to `-inf` and letting the
softmax inside `categorical` normalise once is exact; explicitly dividing by
the surviving mass and then calling a sampler that softmaxes again squares your
probabilities.

The third is ties: `logits < kth_value` keeps *every* token equal to the
$k$-th largest, so a uniform distribution with `top_k=1` keeps the whole
vocabulary. That is the reference behaviour in every production stack, but you
should be able to say why.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def sample_top_k_top_p(key, logits, *, temperature=1.0, top_k=None, top_p=1.0):
    """Draw one token id from a filtered categorical distribution.

    Args:
        key:         a jax.random key
        logits:      (V,) unnormalised scores
        temperature: divide the logits by this before filtering
        top_k:       keep only the k largest logits (None = no top-k)
        top_p:       keep the smallest high-probability set with mass >= top_p

    Returns:
        Scalar int array — the sampled token id.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

logits = jnp.log(jnp.array([0.5, 0.25, 0.125, 0.075, 0.05]))
keys = jax.random.split(jax.random.key(0), 4000)


def hist(**kw):
    toks = jax.vmap(lambda k: sample_top_k_top_p(k, logits, **kw))(keys)
    return jnp.bincount(toks, length=5) / 4000


print("raw            ", hist())
print("temperature 0.5", hist(temperature=0.5), "<- sharpened")
print("temperature 2.0", hist(temperature=2.0), "<- flattened")
print("top_k=2        ", hist(top_k=2), "<- support of size 2")
print("top_p=0.8      ", hist(top_p=0.8), "<- adaptive support")
print("T=0.5, p=0.8   ", hist(temperature=0.5, top_p=0.8),
      "<- nucleus shrinks because temperature ran first")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("topk_sampling")

# hint("topk_sampling")      # stuck? nudge without the answer
# solution("topk_sampling")  # spoiler: the reference implementation